# Lecture: From Video Pixels to Match Report

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Order the full pipeline: video → detect → track → homography → stats → VLM/LLM → report/API (§9 map).
- Explain detection (color masks + centroids, §4), tracking (gate + coasting, §5), and homography (pixels to meters, §6) in plain words.
- Read possession + heatmap (§7) and the VLM/report/API end (§8), and name two silent failures numbers hide: merged identities and coasted speed spikes.


## The big idea

The capstone chains every skill into one sports-analytics factory. **Detection** finds each player and the ball per frame (pixels close to a known kit color form a mask; the mask's average position — the **centroid** — is the location; **recall** = found / (frames × objects)). **Tracking** links those dots across frames (nearest neighbour within a **gate** distance; short gaps are **coasted** by repeating the last spot). A **homography (H)** converts pixel coordinates to pitch meters so distances and speeds are real. **Stats** (distance, top speed, **possession** share, **heatmap** zone counts) plus a vision question-answering check (**VLM stub** vs text-only) feed a templated match **report** served through an **API** dict shaped like real routes.

Real-world analogy: a scoreboard crew. One person spots jerseys each second (detect), another carries names forward on a roster board even when someone is briefly hidden (track + coast), a surveyor maps camera pixels to field yards (homography), statisticians tally distance and possession (stats), and an editor publishes the recap on the club site (report + API). If two teammates share one color, the spotter merges them — the tally looks fine while being wrong, which is why identity matters more than raw counts.


In [ ]:
import numpy as np  # bring in numpy for tracks plus stats
import matplotlib  # bring in matplotlib core to pin the headless backend
matplotlib.use('Agg')  # force headless PNG backend so any machine can render
import matplotlib.pyplot as plt  # bring in pyplot for the pitch demo
np.random.seed(24)  # fix the seed for reproducibility
t = 24  # set frame count matching the lab video
p0 = {'A1': (20.0, 20.0), 'A2': (80.0, 10.0), 'B1': (50.0, 40.0), 'B2': (70.0, 55.0), 'ball': (22.0, 22.0)}  # set start pixels per object
p1 = {'A1': (40.0, 30.0), 'A2': (65.0, 20.0), 'B1': (66.0, 30.0), 'B2': (62.0, 40.0), 'ball': (64.0, 30.0)}  # set end pixels per object
sx, sy = 105.0 / 96, 68.0 / 64  # set meters per pixel: homography scale for this camera
ids = ['A1', 'A2', 'B1', 'B2', 'ball']  # fix the object order
cols = {'A1': 'red', 'A2': 'orange', 'B1': 'blue', 'B2': 'teal', 'ball': 'black'}  # set one plot color per object
pitch = {}  # start the meters-track dict
pairs = list(ids)  # copy the id order for the build loop
start = np.array([p0[k] for k in pairs])  # stack start pixels: 5x2
end = np.array([p1[k] for k in pairs] )  # stack end pixels: 5x2 (extra space kept simple)
print('scene: 5 objects (4 players + ball) over 24 frames at 10fps')  # print the scene size
mid = 52.5  # set the midfield line in meters
ball0x = p0['ball'][0] * sx  # map ball start x to meters
ball1x = p1['ball'][0] * sx  # map ball end x to meters
print('ball starts x', round(ball0x, 1), 'm (left) and ends x', round(ball1x, 1), 'm (right)')  # print the long ball crossing
print('frame-10 dropout is coasted: repeat last spot keeps 24 rows, zero NaN')  # print the continuity trick
print('corner error about 0m: pure-scale homography maps corners exactly')  # print the calibration proof
fig, ax = plt.subplots(figsize=(9, 4.5))  # create a pitch-shaped canvas
ax.set_xlim(0, 105)  # fix full pitch length in meters
ax.set_ylim(0, 68)  # fix full pitch width in meters
ax.set_xlabel('x (m)')  # label x in meters
ax.set_ylabel('y (m)')  # label y in meters
ax.set_title('Toy pitch tracks: A1 A2 B1 B2 + ball (long ball crosses halfway)')  # title the story
ax.axvline(mid, color='gray', linestyle='--', linewidth=1)  # draw the dashed midfield line
fr = np.linspace(0, 1, t)  # build interpolation alphas 0..1 over frames
al = fr[0]  # touch the alpha array to keep the comment habit
ax.plot(p0['A1'][0] + (p1['A1'][0] - p0['A1'][0]) * fr * sx, p0['A1'][1] + (p1['A1'][1] - p0['A1'][1]) * fr * sy, marker='o', ms=3, label='A1', color='red')  # draw the A1 meters path
ax.plot(p0['A2'][0] + (p1['A2'][0] - p0['A2'][0]) * fr * sx, p0['A2'][1] + (p1['A2'][1] - p0['A2'][1]) * fr * sy, marker='o', ms=3, label='A2', color='orange')  # draw the A2 meters path
ax.plot(p0['B1'][0] + (p1['B1'][0] - p0['B1'][0]) * fr * sx, p0['B1'][1] + (p1['B1'][1] - p0['B1'][1]) * fr * sy, marker='o', ms=3, label='B1', color='blue')  # draw the B1 meters path
ax.plot(p0['B2'][0] + (p1['B2'][0] - p0['B2'][0]) * fr * sx, p0['B2'][1] + (p1['B2'][1] - p0['B2'][1]) * fr * sy, marker='o', ms=3, label='B2', color='teal')  # draw the B2 meters path
ax.plot(p0['ball'][0] + (p1['ball'][0] - p0['ball'][0]) * fr * sx, p0['ball'][1] + (p1['ball'][1] - p0['ball'][1]) * fr * sy, marker='o', ms=3, label='ball', color='black')  # draw the ball meters path
ax.legend(fontsize=8)  # show the object legend
ax.grid(True, alpha=0.3)  # add a light grid for reading meters
plt.tight_layout()  # avoid label overlap
plt.show()  # render the pitch


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 spots jerseys, §5 carries names forward, §6 surveys the field, §7–§8 publish the recap.

| Term | One-liner | Section |
|---|---|---|
| Detect / recall | Find objects per frame; recall = found / (frames × objects); lab 1.0000 clean | §4 |
| Centroid | Mean position of a color mask's pixels — the detected center | §4 |
| Track / gate | Link dots across frames; gate (8 px) caps jump per frame | §5 |
| Coast | Bridge short gaps by repeating last spot (frame-10 ball, budget 2) | §5 |
| Homography (H) | Map image pixels → pitch meters (pure scale here); corner error ≈ 0 m | §6 |
| Possession | Nearest player within 6 m of ball each frame; shares + none = 1 | §7 |
| Heatmap | 12×8 zone-visit counts showing where play happened | §7 |
| VLM stub | Rule-based stand-in: which half is the ball in? vision 1.00 vs text 0.625 | §8 |
| Report / API | Templated summary (report.md) + dict shaped like /report + /tracks | §8 |


### §4 Detect and centroid — spotting jerseys by color

**In one sentence: detection thresholds L1 color distance (< 60) per known kit to build one mask per object, and the centroid (mean of mask pixels) is the detected center — clean-video recall is 1.0000 with sub-pixel error.**

Green pitch sits ~200 away from every kit in L1, so discs pop out exactly. But recall counts found-vs-missing, not correct-identity: one mask covering two same-color players still counts as found while its midpoint sits between them. Watch it: a synthetic disc, its mask, and the centroid crosshair.


In [ ]:
import numpy as np  # reuse numpy for the mask math
import matplotlib.pyplot as plt  # reuse pyplot for disc + mask + centroid
np.random.seed(24)  # fix the seed for reproducibility
h, w, r = 32, 32, 5  # set canvas height/width and disc radius
green = np.array([20, 120, 40])  # set the pitch background color
kit = np.array([230, 30, 30])  # set the red kit color to detect
yy, xx = np.mgrid[0:h, 0:w]  # build pixel coordinate grids
m = (xx - 16) ** 2 + (yy - 16) ** 2 <= r * r  # paint the boolean disc mask
fr = np.zeros((h, w, 3)) + green  # start from green pitch
fr[m] = kit  # paint the red disc
d = np.abs(fr - kit).sum(axis=-1)  # score L1 color distance per pixel
mask = d < 60  # threshold the mask (disc passes, green fails by ~200)
ys, xs = np.where(mask)  # collect mask pixel coords
cx, cy = float(xs.mean()), float(ys.mean())  # average them: the centroid
print('centroid:', (round(cx, 2), round(cy, 2)), '(true center 16, 16: sub-pixel)')  # print the localization proof
print('clean recall 1.0: thresholding is exact on clean renders')  # print the detection contract
print('warning: recall cannot see merged identities (one mask, two players)')  # print the silent failure
fig, ax = plt.subplots(1, 2, figsize=(9, 4))  # make two panels: frame + mask
ax[0].imshow(fr.astype(np.uint8))  # show the synthetic frame
ax[0].set_title('Frame: red disc on green')  # title the input
ax[1].imshow(mask, cmap='gray')  # show the threshold mask
ax[1].plot(cx, cy, 'ro', ms=6)  # mark the centroid with a red dot
ax[1].set_title('Mask (L1<60) + centroid dot')  # title the detection
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §5 Track, gate, and coast — carrying names forward

**In one sentence: the tracker greedily links each detection within a gate (8 px max jump per frame), and coasts through short gaps (budget 2) by repeating the last spot — the scripted frame-10 ball dropout leaves 24 rows and zero NaN.**

Color gives identity for free here (one mask per id), so tracking is per-id smoothing, not matching. Coasting preserves continuity but corrupts derivatives: the frozen spot then the catch-up jump inflates top speed past any real kick. Watch it: the frame-dropout timeline (hit…miss…hit) with the coasted bridge.


In [ ]:
import numpy as np  # reuse numpy for the dropout timeline
import matplotlib.pyplot as plt  # reuse pyplot for the timeline
np.random.seed(24)  # fix the seed for reproducibility
t, drop = 24, 10  # set frame count and the scripted ball-dropout frame
gate, coast = 8.0, 2  # set gate px and coast budget in frames
hits = [1] * t  # start with all frames hit
hits[drop] = 0  # hide the ball at frame 10
print('ball hits:', sum(hits), '/', t, '(frame 10 dropped, rest hit)')  # print the dropout pattern
print('coast budget 2 covers the 1-frame gap -> 24 rows, zero NaN')  # print the continuity proof
print('side effect: frozen spot + catch-up jump inflates ball top speed')  # print the derivative artifact
fig, ax = plt.subplots(figsize=(9, 3.0))  # make a flat timeline canvas
ax.set_xlim(-1, t)  # fix room for 24 frames
ax.set_ylim(0, 2)  # fix vertical room
ax.set_xlabel('frame')  # label the x-axis
ax.set_title('Ball timeline: 24 hits, 1 scripted miss at frame 10, coasted')  # title the gap story
ax.set_yticks([])  # hide y ticks: timeline only
xx = list(range(t))  # list frame indices
yy = [1] * t  # lift every marker to one row
cc = ['#16a34a' if h else '#ef4444' for h in hits]  # color green = hit, red = miss
ax.scatter(xx, yy, c=cc, s=60, zorder=3)  # draw one dot per frame
ax.annotate('', xy=(drop + 0.9, 1), xytext=(drop - 0.9, 1), arrowprops=dict(arrowstyle='<->', color='#7c3aed', lw=2))  # draw the purple coast bridge
ax.text(drop, 0.4, 'coast: repeat last spot', ha='center', fontsize=9, color='#7c3aed')  # label the bridge
ax.grid(True, alpha=0.3, axis='x')  # add a light x grid
plt.tight_layout()  # snug the layout
plt.show()  # render the timeline


### §6 Homography H — from pixels to meters

**In one sentence: the homography is the pixel→meter map (here a pure scale: 105/96 in x, 68/64 in y), and corner error (~0 m) proves the calibration by mapping the four image corners onto the four pitch corners.**

Real meters make stats honest: steps in pixels mean nothing until scaled, and top speed needs the 10 fps clock too. Watch it: the corner table (mapped vs expected) plus the error math.


In [ ]:
import numpy as np  # reuse numpy for the corner math
import matplotlib.pyplot as plt  # reuse pyplot for the corner-error bars
np.random.seed(24)  # fix the seed for reproducibility
sx, sy = 105.0 / 96, 68.0 / 64  # set meters per pixel from the pure-scale map
corners = np.array([[0, 0], [96, 0], [96, 64], [0, 64]], dtype=float)  # list the four image corners in px
mapped = corners * np.array([sx, sy])  # map corners to pitch meters
want = np.array([[0, 0], [105, 0], [105, 68], [0, 68]], dtype=float)  # list the expected pitch corners in m
cerr = float(np.abs(mapped - want).max())  # take the max corner error in meters
print('mapped corners (m):', np.round(mapped, 6).tolist())  # print the pixels-became-meters table
print('corner error:', cerr, 'm (exact scale map)')  # print the calibration proof
a1 = np.stack([np.linspace(20 * sx, 40 * sx, 24), np.linspace(20 * sy, 30 * sy, 24)], axis=1)  # build the toy A1 meters track
seg = np.linalg.norm(np.diff(a1, axis=0), axis=1)  # measure per-frame step lengths in meters
dist = float(seg.sum())  # total the distance in meters
top = float(seg.max() / 0.1)  # divide max step by 0.1s per frame: top speed
print('A1: distance', round(dist, 2), 'm, top', round(top, 2), 'm/s at 10fps')  # print stats from meter tracks
fig, ax = plt.subplots(figsize=(7, 4))  # make a figure for per-corner errors
ax.bar(['c0', 'c1', 'c2', 'c3'], np.abs(mapped - want).max(axis=1), color='#16a34a')  # draw error per corner
ax.set_ylabel('corner error (m)')  # label the y-axis
ax.set_title('Calibration: corner error about 0 everywhere')  # title the proof
ax.grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


### §7 Possession and heatmap — who owned the ball, where was play

**In one sentence: possession credits each frame to the nearest player within 6 m of the ball (shares A ≈ 0.17, B ≈ 0.12, none ≈ 0.71 sum to 1), and the heatmap counts all positions into 12×8 pitch zones.**

The long ball starts at A's feet and ends at B's, so both shares are positive with most frames in transition (none). Watch it: the share bars plus a toy zone heatmap.


In [ ]:
import numpy as np  # reuse numpy for shares plus heatmap
import matplotlib.pyplot as plt  # reuse pyplot for bars + heatmap
np.random.seed(24)  # fix the seed for reproducibility
pa, pb = 0.17, 0.12  # set lab possession shares for team A and B
pn = 1.0 - pa - pb  # give the rest to transition frames with none nearby
print('possession: A =', pa, '| B =', pb, '| none =', round(pn, 2), '(sums to 1)')  # print the bookkeeping check
print('6m radius absorbs small centroid wobble; long ball keeps both shares > 0')  # print the rule + contract
heat = np.zeros((8, 12))  # start the 12x8 zone grid (transposed for imshow)
heat[2, 2] = 9  # park A1 start zone visits
heat[5, 9] = 8  # park B2 end zone visits
heat[3, 5] = 12  # park midfield traffic visits
heat = heat + 1  # pad every zone with one visit
print('heatmap zones: 12x8 = 96 cells, peak 13 visits at midfield')  # print the zone summary
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: shares + heatmap
ax[0].bar(['team A', 'team B', 'none'], [pa, pb, pn], color=['#ef4444', '#3b82f6', '#9ca3af'])  # draw possession shares
ax[0].set_ylim(0, 1.0)  # fix the share range
ax[0].set_ylabel('share of frames')  # label the y-axis
ax[0].set_title('Possession: nearest player within 6m')  # title the rule
ax[0].grid(True, alpha=0.3, axis='y')  # add a light grid
im = ax[1].imshow(heat, extent=[0, 105, 0, 68], aspect='auto')  # draw the zone heatmap over pitch extent
ax[1].set_xlabel('x (m)')  # label x
ax[1].set_ylabel('y (m)')  # label y
ax[1].set_title('Occupancy heatmap (12x8 zones)')  # title the panel
fig.colorbar(im, ax=ax[1], label='visits')  # add the visit-count scale
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §8 VLM, report, and API — vision answers, text serves

**In one sentence: the VLM stub answers ball-half QA from pixels (1.00) while the text-only arm constantly guesses left (0.625 = the left share), the templated report fills measured stats into sentences, and the API dict (`report`, `tracks`, `top_speed`, `possession`) is the same schema real routes would serve.**

The constant guess scores the left share, not a coin flip — vision's 1.00 still proves pixels resolve what the blind guess cannot. Watch it: the QA bars plus the response-schema print.


In [ ]:
import numpy as np  # reuse numpy for the QA split
import matplotlib.pyplot as plt  # reuse pyplot for the QA bars
np.random.seed(24)  # fix the seed for reproducibility
av, at = 1.00, 0.625  # set ball-half QA: vision exact vs text-only constant-left guess
print('ball-half QA: vision =', av, 'vs text-only =', at, '(constant guess = left share)')  # print the multimodality gap
api = {'report': 'results/report.md', 'tracks': 'results/tracks.csv', 'top_speed': 'fastest player row', 'possession': {'A': 0.17, 'B': 0.12}}  # shape the API payload
print('API keys:', sorted(api.keys()), '(same schema a FastAPI route serves)')  # print the serving shape
print('report line: Possession team A 17%, team B 12% over 24 frames at 10fps')  # print one pixels-to-sentence line
fig, ax = plt.subplots(figsize=(6, 4))  # make a figure for the QA bars
ax.bar(['vision', 'text-only'], [av, at], color=['#16a34a', '#9ca3af'])  # draw the two arms
ax.set_ylim(0, 1.1)  # fix the range with headroom
ax.set_ylabel('ball-half accuracy')  # label the y-axis
ax.set_title('VLM 1.00 vs blind 0.625: pixels resolve halves')  # title the gap
ax.grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


## How it all fits together — the system map

One frame walks the full-curriculum map. **Video → detect** (§4): 96×64 px, color masks, centroids, recall 1.0000. **Track** (§5): 8 px gate, 2-frame coast over the frame-10 dropout. **H** (§6): scale to meters, corner error ≈ 0. **Stats** (§7): distance, top speed at 10 fps, possession, heatmap. **VLM/LLM → report/API** (§8): 1.00 vs 0.625 QA, templated sentences, route-shaped dict. The map pins every course skill; the numbers trace one frame from pixels to a report line.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(12, 3.5))  # create a wide figure for the full map
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 16)  # fix horizontal placement range
ax.set_ylim(0, 4)  # fix vertical placement range
ax.text(8, 3.6, 'CAPSTONE: video->detect->track->H->stats->VLM/LLM->report/API', ha='center', fontsize=9, fontweight='bold')  # full-pipeline banner line
ax.text(1.2, 2.4, 'video\n§4', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the video box
ax.text(3.2, 2.4, 'detect\n§4', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the detect box
ax.text(5.2, 2.4, 'track\n§5', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the track box
ax.text(7.2, 2.4, 'H\n§6', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the homography box
ax.text(9.2, 2.4, 'stats\n§7', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the stats box
ax.text(11.4, 2.4, 'VLM/LLM\n§8', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the model box
ax.text(14, 2.4, 'report/API\n§8', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the serving box
ax.text(8, 1.2, 'recall 1.0 | coast frame-10 | corner err ~0 | poss A 0.17 B 0.12 | QA 1.00 vs 0.625', ha='center', fontsize=8, fontweight='bold', color='darkred')  # red = metric banner
ax.text(8, 0.5, 'every track skill behind one schema: report.md + tracks.csv + vlm.csv', ha='center', fontsize=8, style='italic')  # artifact reminder line
ax.set_title('System map: the full-curriculum pipeline in seven hops')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the pixels-to-report trace
np.random.seed(24)  # fix the seed for reproducibility
sx, sy = 105.0 / 96, 68.0 / 64  # reuse the homography scale
px = (22.0, 22.0)  # take frame-0 ball pixels (at A1 feet)
m = (round(px[0] * sx, 2), round(px[1] * sy, 2))  # map pixels to meters
print('frame 0 ball pixels:', px, '-> meters:', m, '(left half: x < 52.5)')  # print the H step
half = 'left' if m[0] < 52.5 else 'right'  # answer the ball-half question from meters
print('VLM half answer:', half, '(vision 1.00; blind text-only guesses left)')  # print the QA step
print('possession: nearest player within 6m -> team A frame (share A 0.17)')  # print the stats step
print('report line: Ball starts left at team A feet; possession A 17% / B 12%.')  # print the pixels-to-sentence line
print('API serves it: GET /report -> report.md, GET /tracks -> tracks.csv')  # print the serving step


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above (§9):

1. **Goal + pipeline** — recite video, detect, track, homography, stats, VLM, report, API before any code (§4–§8).
2. **Render** — the seeded 24-frame scene (4 players + ball, linear motion, radius-3 discs); this lecture's pitch plot is its source data (§4).
3. **Detect** — the L1 color-distance < 60 mask and centroid mean; recall 1.0000, sub-pixel error — then recall's blindness to merged identities (§4).
4. **Track + homography** — 8 px gate, 2-frame coast, scripted frame-10 dropout (§5); pure-scale H with corner error ≈ 0; all tracks become PITCH meters (§6).
5. **Stats + VLM + report** — distance/top-speed at 10 fps, 6 m possession (A 0.17, B 0.12), heatmap (§7); ball-half QA 1.00 vs 0.625; templated report.md + API dict (§8).
6. **Noise ablation + figure** — sigma=25 noise keeps recall ≥0.90 (the notebook asserts nrec≥0.90, not 1.00 — colors are far apart so noise costs little) (colors far apart); outputs land in `tracks.csv`, `summary.csv`, `vlm.csv`, `report.md`, `capstone.png`.
7. **Cleanup/done** — the course map: swap render/detect/track/stubs for video, YOLO, DeepSORT, served models behind P23's gateway, priced with P22's math.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Detection recall is 1.0000, yet two same-color players were once merged into one midpoint track. How (§4)?**

*Answer: recall counts found-vs-missing, not correct-identity: one mask covering two players still counts as found while its centroid sits between them. Identity errors need appearance features, not just counts.*

2. **The coasted ball shows top speed past any real kick. What happened (§5)?**

*Answer: coasting repeats the last spot during the dropout, then jumps two frames at once on recovery. Continuity is preserved (no NaN, 24 rows) but the derivative spikes — continuity is not differentiability.*

3. **Why does the text-only arm score 0.625 on ball-half QA instead of about 0.50 (§8)?**

*Answer: it constantly guesses left, so its score equals the left share of frames, not a coin flip. The vision arm's 1.00 still proves pixels resolve what the constant guess cannot — the §9 trace answers the same half question from meters.*
